# A mini investigation: is the world converging?

Here's a real question, not a made-up one: since 1950, has life
expectancy around the world been **converging** — poorer countries
catching up to richer ones — or has the gap been staying the same, or
even widening?

You could guess. This notebook finds out, using real data from
[Our World in Data](https://ourworldindata.org/life-expectancy)
(`data/life-expectancy.csv`, shipped with dewlab): one row per country
per year from 1950 to 2023, each with that country's average life
expectancy at birth.

An investigation like this has a shape: load the data, look at the
whole picture first, then narrow in, then ask what the pattern might
mean. Let's follow it.

In [ ]:
df = await load_csv("life-expectancy.csv")
show_table(df.head())

Three columns: `country`, `year`, `life_expectancy`. About two dozen
`country` values aren't really countries at all — `"World"`,
`"Europe"`, `"High-income countries"` — regional and income-group
averages OWID includes alongside individual countries, plus the parts
of one country (`"Scotland"`) and one that no longer exists (`"USSR"`). For a
country-by-country comparison those would skew things (a continent
isn't a country, and counting it as one double-counts everything
inside it), so the first real step is deciding what to leave out.

In [ ]:
not_countries = {
    "World", "Africa", "Americas", "Asia", "Europe", "Oceania", "Northern America",
    "Latin America and the Caribbean",
    "High-income countries", "Upper-middle-income countries", "Middle-income countries",
    "Lower-middle-income countries", "Low-income countries",
    "High-and-upper-middle-income countries", "Low-and-middle-income countries",
    "Low-and-Lower-middle-income countries", "No income group available",
    "More developed regions", "Less developed regions",
    "Less developed regions, excluding China",
    "Less developed regions, excluding least developed countries",
    "Least developed countries", "Land-locked Developing Countries (LLDC)",
    "Small Island Developing States (SIDS)",
    "England and Wales", "Scotland", "Northern Ireland",   # parts of the United Kingdom
    "USSR",                                                # counted again in its successors
}
countries = df[~df["country"].isin(not_countries)]
print(f"{df['country'].nunique()} entities in total, {countries['country'].nunique()} of them real countries")

That's the kind of unglamorous step every real dataset needs somewhere
— deciding what actually counts as one of the things you're comparing.
Skip it and every average below would be quietly wrong.

## The whole picture: has the average changed?

Group by year, average `life_expectancy` across every country that year,
and plot it.

In [ ]:
yearly_mean = countries.groupby("year")["life_expectancy"].mean()

import matplotlib.pyplot as plt
plt.plot(yearly_mean.index, yearly_mean.values)
plt.title("Average life expectancy across all countries, by year")
plt.xlabel("Year")
plt.ylabel("Years")
plt.show()

A clear, steady climb — no real surprise there; that part of the story
is well known. The actual question this notebook is asking is more
specific than "did it go up": did every country climb *together*, or
did some pull ahead while others were left behind? The average alone
can't answer that — a rising average is exactly what you'd see either
way. You need the *spread*, not just the *centre*.

## The spread: how different are countries from each other?

The standard deviation of `life_expectancy` in a given year is one
number that measures exactly this — how far, on average, individual
countries sit from that year's mean. A big number means countries are
scattered far apart; a small number means they're clustered close
together.

In [ ]:
yearly_spread = countries.groupby("year")["life_expectancy"].std()

plt.plot(yearly_spread.index, yearly_spread.values)
plt.title("How spread out countries' life expectancies are, by year")
plt.xlabel("Year")
plt.ylabel("Standard deviation (years)")
plt.show()

Read that trend before moving on — does the spread grow, shrink, or
stay flat across the seven decades? That answer is your finding, before
any number below confirms it.

## Naming what you just saw

If that line falls, the gap between countries has been **narrowing** —
convergence. Poorer countries, on average, have been gaining faster
than richer ones already near the biological ceiling of what life
expectancy can reach. Check the two ends of the data to see the actual
size of it:

In [ ]:
print("1950:", round(yearly_mean[1950], 1), "average, spread", round(yearly_spread[1950], 1))
print("2023:", round(yearly_mean[2023], 1), "average, spread", round(yearly_spread[2023], 1))

Both numbers moved — the average rose by more than twenty years, and
the spread shrank by about two-fifths. That combination is the finding: not
just "people live longer now," which is the part everyone already
expects, but "the gap between countries has been closing while it
happened," which is the part worth actually checking rather than
assuming.

## Who's furthest from the average, in the most recent year?

`nsmallest` and `nlargest` pull out the extremes directly, no sorting
the whole table by hand.

In [ ]:
latest_year = countries["year"].max()
latest = countries[countries["year"] == latest_year]
show_table(latest.nsmallest(5, "life_expectancy"), caption=f"Lowest, {latest_year}")
show_table(latest.nlargest(5, "life_expectancy"), caption=f"Highest, {latest_year}")

## Your turn

Pick one country from each table above — one from the bottom, one from
the top — and plot its own `life_expectancy` over the full range
since 1950, on the same axes, the way the SQL notebook plotted Ireland alone.
(`hint:` `countries[countries["country"] == "..."]` gets you one
country's rows; call `plt.plot()` twice, once per country, before the
one `plt.show()`, to get both lines on the same chart.)

Then look at the shapes, not just the end points. Did the country that
started furthest behind climb *faster* than the one that started ahead —
which is what "converging" actually requires — or did both simply rise
in parallel, with the gap between them never really closing? The
first chart in this notebook already answered that question for the
world as a whole. This one answers it for the two countries you picked,
and does either of the two answers surprise you?

In [ ]:
# Your turn — two countries, one chart, since 1950.


## Looking back

This is what a real data investigation looks like: not "run a script and
get an answer," but a sequence of small, honest decisions — what to
exclude, which single number actually measures the question you're
asking, which two things are worth comparing next. None of the code
above was complicated. The thinking about *what to compute* was the
actual work.

**Source**: Our World in Data, ["Life expectancy at birth"](https://ourworldindata.org/grapher/life-expectancy)
(compiled from Riley (2005), Zijdeman et al. (2015), the Human Mortality
Database and the UN World Population Prospects). Licensed
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The figures
above are from the copy saved on 26 September 2026; a newer copy may
give slightly different ones.